# bitladder: MNIST bit ladder in Colab or Kaggle

One straight run: Runtime → Run all (GPU, a T4 is enough; no dataset to fetch, MNIST downloads itself).
Sweep: 5 widths × 12 rungs (fp32, fp16, uniform 8/4/2, ternary, binary, each float-activation and
8-bit activation) × 10 epochs = 60 trainings, about 1–2 h on a T4. One `run_once` per width, results pushed
to the private results repo after each width, so a disconnect loses at most one width: re-run the notebook
and it continues.

**Colab**: Runtime → T4 GPU; Secrets (key icon) → `RESULTS_TOKEN` with notebook access.
**Kaggle**: File → Import notebook → GitHub URL of this file; Settings → Accelerator GPU, Internet ON;
Add-ons → Secrets → `RESULTS_TOKEN`.

The result (`results/mnist_ladder.csv` / `.md`, one JSON per training) goes to the `sandbox_mnist` section
of the docs only; the audio track stays the main one.

In [ ]:
# 1. Code (Colab or Kaggle). Experiments go through run_once(): a crash STOPS the cell, finished runs are skipped on a re-run.
import os, sys
ON_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
WORK = '/kaggle/working' if ON_KAGGLE else '/content'
REPO = 'https://github.com/Vzzzz/bitladder.git'
if not os.path.exists(f'{WORK}/bitladder'):
    !git clone -q {REPO} {WORK}/bitladder
%cd {WORK}/bitladder
!git pull -q --ff-only && git log -1 --oneline
!pip install -q -e . 2>&1 | grep -iE 'error|failed' || true
for m in [m for m in sys.modules if m == 'bitladder' or m.startswith('bitladder.')]:
    del sys.modules[m]   # a kernel that imported the old code before `git pull` would keep using it
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
from bitladder.notebook import run, run_once, done   # raises ImportError if the install above failed

In [ ]:
# S. Results sync: after every width, CSV/MD/JSON go to the private results repo; a new runtime gets them back and
# continues. Token: fine-grained PAT in the notebook secrets as RESULTS_TOKEN (Colab: key icon, notebook access ON;
# Kaggle: Add-ons -> Secrets), Repository access -> bitladder-results, Contents: read and write.
# Fails hard if the sync cannot start; set ALLOW_NO_SYNC = True only to run on purpose without it.
import datetime
from bitladder.results_sync import ResultsSync
RESULTS_REPO = 'Vzzzz/bitladder-results'
LABEL = f"{'kaggle' if ON_KAGGLE else 'colab'}-{datetime.date.today()}"
ALLOW_NO_SYNC = False

def _secret(name):
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    from google.colab import userdata
    return userdata.get(name)

try:
    sync = ResultsSync(RESULTS_REPO, _secret('RESULTS_TOKEN'), LABEL, work_dir='/tmp')
    print('results sync on ->', RESULTS_REPO, '/', LABEL)
    print('restored from the repo:', sync.restore('results'), 'file(s)')
except Exception as e:
    msg = f'{type(e).__name__}: {str(e)[:300]}'
    if not ALLOW_NO_SYNC:
        raise RuntimeError('results sync could not start: ' + msg) from e
    print('results sync OFF:', msg)
    class _NoSync:
        def push(self, *a, **k):
            return 0
        def restore(self, *a, **k):
            return 0
    sync = _NoSync()

In [ ]:
# 2. Smoke: one width, one epoch, 512 samples, every rung (~1 min). Catches an environment problem before the long sweep.
run('python sandbox_mnist/sweep.py --epochs 1 --widths 4 --act-bits 0 8 --subset 512 --quiet --out /tmp/mnist_smoke '
    '--csv /tmp/mnist_smoke/ladder.csv', show=('wrote',))

In [ ]:
# 3. The ladder, one run_once per width (~15-25 min each on a T4). Re-run after a disconnect: finished widths are skipped.
WIDTHS = (4, 6, 8, 12, 16)
for w in WIDTHS:
    name = f'mnist_ladder_w{w}.csv'
    run_once(f'python sandbox_mnist/sweep.py --epochs 10 --widths {w} --act-bits 0 8 --quiet --csv results/{name}',
             name, show=('wrote',))
    sync.push('results')

In [ ]:
# 4. Merge the per-width tables into results/mnist_ladder.csv / .md (Pareto front: bytes vs accuracy) and push.
import pandas as pd
sys.path.insert(0, 'sandbox_mnist')
from sweep import pareto, to_markdown
df = pd.concat([pd.read_csv(f'results/mnist_ladder_w{w}.csv') for w in WIDTHS], ignore_index=True)
df.to_csv('results/mnist_ladder.csv', index=False)
rows = df.to_dict('records')
md = to_markdown(rows, pareto(rows))
open('results/mnist_ladder.md', 'w').write(md)
print(md)
sync.push('results')
!tar -czf {WORK}/mnist_results.tgz results/mnist_ladder*.csv results/mnist_ladder*.md results/mnist_*.json && ls -la {WORK}/mnist_results.tgz